# Build LDA Tables

This notebook fits an LDA topic model on the constitution corpus. It is configured to use the same bag structure as the derived-table pipeline, so article-level bags can be modeled directly and then summarized back to one row per constitution for interpretation.

That choice is justified because the corpus contains more than 33,000 article-level documents, and article-level bags preserve local thematic structure better than collapsing each constitution into one very large bag. The tradeoff is that once topic weights are averaged back to the constitution level, within-constitution variation across articles becomes less visible.


## Inputs and Outputs

**Input files**
- `corpus.csv`
- `lib.csv`

**Output files**
- `lda_dtm.parquet`
- `lda_theta.parquet`
- `lda_phi.parquet`
- `lda_topics.parquet`

The notebook keeps the main HW08 choices of noun-only bags, a count-based `CountVectorizer`, and a 20-topic LDA model. It mirrors the PCA workflow by fitting on the active bag level and then aggregating topic weights back to the constitution level for summary tables and interpretation. In the final project write-up, the explicit CSV requirement applies to the core `LIB`, `CORPUS`, and `VOCAB` tables; these downstream LDA outputs are saved as Parquet.

In [1]:
# Maintainer note: this topic-model notebook is adapted from the HW08 workflow in `nbz3de-m08-hw.ipynb`.
# Keep the homework provenance here in code comments instead of the final-project-facing documentation.
from pathlib import Path

import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation as LDA
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

try:
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio
    pio.renderers.default = 'plotly_mimetype'
except ModuleNotFoundError:
    px = None
    go = None
    pio = None

CORPUS_CSV = Path('corpus.csv')
LIB_CSV = Path('lib.csv')

LDA_DTM_PARQUET = Path('lda_dtm.parquet')
LDA_THETA_PARQUET = Path('lda_theta.parquet')
LDA_PHI_PARQUET = Path('lda_phi.parquet')
LDA_TOPICS_PARQUET = Path('lda_topics.parquet')

BAG_LEVELS = {
    'DOCS': ['country_id'],
    'ARTICLES': ['country_id', 'article_n'],
    'CLAUSES': ['country_id', 'article_n', 'clause_n'],
}
LDA_BAG_NAME = 'ARTICLES'
BAG_COLS = BAG_LEVELS[LDA_BAG_NAME]
POS_PATTERN = r'^NNS?$'
CUSTOM_LEGAL_STOP_TERMS = {
    'article', 'articles', 'chapter', 'chapters', 'paragraph', 'paragraphs',
    'part', 'parts', 'provision', 'provisions', 'section', 'sections',
    'shall', 'subsection', 'subsections'
}
COUNT_STOP_WORDS = sorted(set(ENGLISH_STOP_WORDS).union(CUSTOM_LEGAL_STOP_TERMS))

MAX_DF = 0.80
MIN_DF = 10
MAX_FEATURES = 4000
N_TOPICS = 20
N_TOP_TERMS = 7
TNAMES = [f"T{str(i).zfill(len(str(N_TOPICS)))}" for i in range(N_TOPICS)]

## Load Inputs

The corpus already stores normalized `term_str` values and part-of-speech tags, so we can reproduce the HW08 noun filter directly on the project data. The bag level is configurable so the notebook can model constitutions, articles, or clauses using the same token source. Because `term_str` is a normalized field, some tokens are intentionally left blank during corpus construction; this notebook drops missing normalized terms before joining document strings so missing values do not become the literal token `nan` in LDA.

In [2]:
# Load only the token fields needed for noun filtering and bag reconstruction.
TOKENS = pd.read_csv(
    CORPUS_CSV,
    usecols=['country_id', 'article_n', 'clause_n', 'pos', 'term_str']
)
# Keep LIB keyed by country_id so later summary and visualization joins stay simple.
LIB = pd.read_csv(LIB_CSV).set_index('country_id')

# Drop missing normalized terms so blank values do not become literal tokens in the model input.
TOKENS = TOKENS[TOKENS['term_str'].notna() & (TOKENS['term_str'] != '')].copy()

# Rebuild one noun-only document string per active bag for CountVectorizer.
DOCS = (
    TOKENS[TOKENS['pos'].str.match(POS_PATTERN, na=False)]
    .groupby(BAG_COLS)['term_str']
    .apply(lambda x: ' '.join(x.astype(str)))
    .to_frame('doc_str')
)
DOCS['noun_count'] = DOCS['doc_str'].str.split().str.len()

print(f'Token rows: {len(TOKENS):,}')
print(f'LDA bag level: {LDA_BAG_NAME} -> {BAG_COLS}')
print(f'Documents in LDA bag level: {len(DOCS):,}')
print(f'Min noun count per document: {DOCS.noun_count.min():,}')
print(f'Median noun count per document: {DOCS.noun_count.median():,.0f}')
DOCS.head()

Token rows: 3,839,736
LDA bag level: ARTICLES -> ['country_id', 'article_n']
Documents in LDA bag level: 33,220
Min noun count per document: 1
Median noun count per document: 14


doc_str  \
country_id  article_n                                                      
Afghanistan 1          name disciples followers people divine religio...   
            2                                          independent state   
            3          religion religion faiths bounds law exercise p...   
            4                             law tenets provisions religion   
            5          sovereignty nation representatives nation indi...   

                       noun_count  
country_id  article_n              
Afghanistan 1                  58  
            2                   2  
            3                   8  
            4                   4  
            5                  17

## Vectorize Documents

This step builds a count-based document-term matrix from noun-only document strings with the HW08-style `CountVectorizer` filter. Because this workflow does not use the reduced project `VOCAB` directly, it also adds the same constitution-specific legal boilerplate terms used in `build_vocab.ipynb` to the stopword list. When the bag level is article- or clause-level, the matrix keeps that finer-grained index.

In [3]:
# Fit the HW08-style count vectorizer with the project's added legal boilerplate stop terms.
count_engine = CountVectorizer(
    max_df=MAX_DF,
    min_df=MIN_DF,
    max_features=MAX_FEATURES,
    stop_words=COUNT_STOP_WORDS,
)

# Transform the bag strings into a sparse count matrix, then materialize it as a labeled DataFrame.
count_model = count_engine.fit_transform(DOCS['doc_str'])
TERMS = count_engine.get_feature_names_out()

DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)
if isinstance(DTM.index, pd.MultiIndex):
    DTM.index.names = BAG_COLS
else:
    DTM.index.name = BAG_COLS[0]

print(f'DTM shape: {DTM.shape}')
print(f'Vocabulary size after filtering: {len(TERMS):,}')
print(f'Custom legal stop terms applied: {len(CUSTOM_LEGAL_STOP_TERMS)}')
DTM.iloc[:5, :10]

DTM shape: (33220, 3502)
Vocabulary size after filtering: 3,502
Custom legal stop terms applied: 15


abandonment  abeyance  abide  abilities  ability  \
country_id  article_n                                                     
Afghanistan 1                    0         0      0          0        0   
            2                    0         0      0          0        0   
            3                    0         0      0          0        0   
            4                    0         0      0          0        0   
            5                    0         0      0          0        0   

                       abolition  abroad  abrogation  absence  absences  
country_id  article_n                                                    
Afghanistan 1                  0       0           0        0         0  
            2                  0       0           0        0         0  
            3                  0       0           0        0         0  
            4                  0       0           0        0         0  
            5                  0       0           0        0         0

## Fit LDA and Build Topic Tables

The main outputs follow the HW08 pattern:
- `THETA`: bag-by-topic weights at the configured modeling level
- `PHI`: topic-by-term weights
- `TOPICS`: top terms and overall topic prevalence summarized at the constitution level

In [4]:
# Fit the LDA model on the bag-term matrix and keep both bag-level and constitution-level outputs.
topic_engine = LDA(n_components=N_TOPICS, random_state=42)
topic_model = topic_engine.fit_transform(DTM)

THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
if isinstance(THETA.index, pd.MultiIndex):
    THETA.index.names = BAG_COLS
else:
    THETA.index.name = BAG_COLS[0]
THETA.columns.name = 'topic_id'

PHI = pd.DataFrame(topic_engine.components_, index=TNAMES, columns=DTM.columns)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'

# Aggregate back to one row per country_id when the model is fit on article or clause bags.
if BAG_COLS == ['country_id']:
    THETA_COUNTRY = THETA.copy()
else:
    THETA_COUNTRY = THETA.reset_index().groupby('country_id')[TNAMES].mean()
THETA_COUNTRY.index.name = 'country_id'
THETA_COUNTRY.columns.name = 'topic_id'

# Summarize each topic by its strongest terms and overall prevalence across constitutions.
TOPICS = (
    PHI.stack()
    .groupby('topic_id')
    .apply(lambda x: ' '.join(x.sort_values(ascending=False).head(N_TOP_TERMS).reset_index().term_str))
    .to_frame('top_terms')
)
TOPICS['doc_weight_mean'] = THETA_COUNTRY.mean(axis=0)
TOPICS['dominant_doc_count'] = THETA_COUNTRY.idxmax(axis=1).value_counts().reindex(TNAMES, fill_value=0)
TOPICS = TOPICS.sort_values('doc_weight_mean', ascending=False)

TOPICS

,top_terms,doc_weight_mean,dominant_doc_count
topic_id,,,
T02,law laws courts matters jurisdiction decisions...,0.079318,48
T13,office member election members term years person,0.073307,16
T15,rights people freedoms law citizens principles...,0.071653,29
T18,development education law resources services h...,0.063536,24
T04,office person functions accordance advice appo...,0.060593,30
T00,days period law date time day force,0.057705,5
T06,duties functions law exercise members accordan...,0.055221,3
T10,members majority vote votes election candidate...,0.054429,7
T17,government service power law authority powers ...,0.053186,11


## Inspect Topic Strength by Country

This helper creates a compact table of the strongest constitutions for each topic. If the model was fit on article or clause bags, the notebook first averages topic weights back to one row per `country_id` before ranking them.

That summary is useful for interpretation, but it should be read as a constitution-level average rather than proof that a topic is distributed evenly across all articles in a constitution.


In [5]:
def build_top_docs(theta: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    # Collect the highest-weight constitutions for each topic into one inspection table.
    frames = []
    for topic_id in theta.columns:
        top_docs = (
            theta[topic_id]
            .sort_values(ascending=False)
            .head(top_n)
            .reset_index()
            .rename(columns={topic_id: 'topic_weight'})
        )
        top_docs.insert(0, 'topic_id', topic_id)
        top_docs.insert(1, 'rank', range(1, len(top_docs) + 1))
        frames.append(top_docs)
    return pd.concat(frames, ignore_index=True)


TOP_DOCS = build_top_docs(THETA_COUNTRY, top_n=5)
TOP_DOCS.head(20)

,topic_id,rank,country_id,topic_weight
0,T00,1,Ireland,0.156466
1,T00,2,Bahrain,0.146738
2,T00,3,India,0.141740
3,T00,4,Seychelles,0.132351
4,T00,5,Canada,0.125618
5,T01,1,Brunei,0.125764
6,T01,2,Malaysia,0.100407
7,T01,3,Marshall Islands,0.092639
8,T01,4,Bangladesh,0.085490
9,T01,5,St. Kitts and Nevis,0.082274


## LDA + PCA Visualization

This section applies PCA to the constitution-level `THETA` table with topics treated as observations. Each topic is plotted in the space opened by the first two principal components. Point size is based on mean topic weight across constitutions, and point color is based on the dominant `region_compressed` category in `LIB` after weighting countries by topic strength.

The color is therefore an interpretive overlay rather than a claim that a topic belongs exclusively to one region. It identifies which metadata category contributes the most weight to a topic on average after constitution-level topic weights have already been aggregated.


In [6]:
LDA_PCA_COLOR_COLS = [
    'region_compressed',
    'v2x_regime_cat',
    'v2x_freexp_altinf_cat',
    'v2x_rule_cat',
    'year_created',
    'year_amended',
]

DISPLAY_LABELS = {
    'region_compressed': 'Region',
    'v2x_regime_cat': 'Regime Type',
    'v2x_freexp_altinf_cat': 'Media Freedom',
    'v2x_rule_cat': 'Rule of Law',
    'year_created': 'Year Created',
    'year_amended': 'Year Amended',
}

LDA_PCA_REGION_HTML = Path('lda_pca_region.html')
LDA_PCA_REGION_TABLE_HTML = Path('lda_pca_region_table.html')
LDA_PCA_REGION_TABLE_CSV = Path('lda_pca_region_table.csv')

def build_topic_pca_table(theta_country: pd.DataFrame, topics: pd.DataFrame, lib: pd.DataFrame, color_col: str):
    # Treat topics as observations so PCA shows which topics vary together across constitutions.
    if color_col not in lib.columns:
        raise KeyError(f'{color_col} is not present in LIB.')
    topic_pca = PCA(n_components=2)
    coords = topic_pca.fit_transform(theta_country.T)
    topic_table = pd.DataFrame(coords, index=theta_country.columns, columns=['PC0', 'PC1'])
    topic_table.index.name = 'topic_id'
    topic_table['mean_doc_weight'] = theta_country.mean(axis=0)
    topic_table['top_terms'] = topics.reindex(topic_table.index)['top_terms']
    lib_doc = lib.drop_duplicates(subset=['country_id']).set_index('country_id')
    # Normalize category totals by category size so larger groups do not dominate by count alone.
    weighted = theta_country.join(lib_doc[[color_col]], how='left')
    weighted = weighted.dropna(subset=[color_col])
    dominant_values = {}
    for topic_id in theta_country.columns:
        grouped = weighted.groupby(color_col)[topic_id]
        adjusted = (grouped.sum() / grouped.count()).sort_values(ascending=False)
        dominant_values[topic_id] = adjusted.index[0] if len(adjusted) else 'Unknown'
    topic_table[color_col] = pd.Series(dominant_values)
    return topic_table.reset_index(), topic_pca.explained_variance_ratio_


def build_topic_terms_table(topic_pca: pd.DataFrame, color_col: str, top_n: int = 20) -> pd.DataFrame:
    table = topic_pca[['topic_id', 'top_terms', color_col, 'mean_doc_weight']].copy()
    table = table.sort_values('mean_doc_weight', ascending=False).head(top_n)
    table['mean_doc_weight'] = table['mean_doc_weight'].map(lambda x: f'{x:.4f}')
    return table.rename(columns={
        'topic_id': 'Topic',
        'top_terms': 'Top terms',
        color_col: DISPLAY_LABELS.get(color_col, color_col.replace('_', ' ').title()),
        'mean_doc_weight': 'Mean weight',
    })


def build_topic_terms_table_figure(table_df: pd.DataFrame, title: str):
    table_height = max(320, 80 + 24 * len(table_df))
    fig = go.Figure(
        data=[
            go.Table(
                header=dict(
                    values=list(table_df.columns),
                    align='left',
                    fill_color='#D9E6F2',
                    font=dict(size=11),
                ),
                cells=dict(
                    values=[table_df[col] for col in table_df.columns],
                    align='left',
                    fill_color='white',
                    font=dict(size=10),
                    height=24,
                ),
            )
        ]
    )
    fig.update_layout(
        title=title,
        height=table_height,
        margin=dict(t=50, r=30, b=20, l=30),
    )
    return fig


TOPIC_PCA_TABLES = {}
for color_col in LDA_PCA_COLOR_COLS:
    topic_pca, topic_pca_explained = build_topic_pca_table(THETA_COUNTRY, TOPICS, LIB.reset_index(), color_col)
    topic_pca['size_for_plot'] = topic_pca['mean_doc_weight'] * 1200
    TOPIC_PCA_TABLES[color_col] = (topic_pca, topic_pca_explained)

if px is None or go is None:
    raise ModuleNotFoundError('plotly is required for the LDA + PCA visualization but is not installed in this environment.')

LDA_PCA_FIGS = {}
LDA_PCA_TABLE_FIGS = {}
for color_col, (topic_pca, topic_pca_explained) in TOPIC_PCA_TABLES.items():
    title_label = DISPLAY_LABELS.get(color_col, color_col.replace('_', ' ').title())
    scatter_fig = px.scatter(
        topic_pca,
        x='PC0',
        y='PC1',
        text='topic_id',
        hover_name='topic_id',
        hover_data={
            'top_terms': True,
            'mean_doc_weight': ':.4f',
            color_col: True,
            'size_for_plot': False,
        },
        size='size_for_plot',
        color=color_col,
        title=f'LDA Topics in PCA Space of THETA colored by {title_label}',
        height=700,
    )
    scatter_fig.update_traces(textposition='top center')
    topic_terms_table = build_topic_terms_table(topic_pca, color_col)
    table_fig = build_topic_terms_table_figure(
        topic_terms_table,
        f'Topic Terms Table for {title_label}',
    )
    scatter_fig.show()
    table_fig.show()
    LDA_PCA_FIGS[color_col] = scatter_fig
    LDA_PCA_TABLE_FIGS[color_col] = table_fig

TOPIC_PCA_REGION_YEAR = TOPIC_PCA_TABLES['region_compressed'][0].merge(
    TOPIC_PCA_TABLES['year_created'][0][['topic_id', 'year_created']],
    on='topic_id',
    how='left',
)

scatter_fig_region_year = px.scatter(
    TOPIC_PCA_REGION_YEAR,
    x='PC0',
    y='PC1',
    text='topic_id',
    hover_name='topic_id',
    hover_data={
        'top_terms': True,
        'mean_doc_weight': ':.4f',
        'region_compressed': True,
        'year_created': True,
        'size_for_plot': False,
    },
    size='size_for_plot',
    color='year_created',
    symbol='region_compressed',
    title='LDA Topics in PCA Space: color = Year Created, shape = Region',
    height=700,
)
scatter_fig_region_year.update_traces(textposition='top center')
region_year_table = TOPIC_PCA_REGION_YEAR[['topic_id', 'top_terms', 'region_compressed', 'year_created', 'mean_doc_weight']].copy()
region_year_table = region_year_table.sort_values('mean_doc_weight', ascending=False).head(20)
region_year_table['mean_doc_weight'] = region_year_table['mean_doc_weight'].map(lambda x: f'{x:.4f}')
region_year_table = region_year_table.rename(columns={
    'topic_id': 'Topic',
    'top_terms': 'Top terms',
    'region_compressed': 'Region',
    'year_created': 'Year created',
    'mean_doc_weight': 'Mean weight',
})
LDA_PCA_TABLE_REGION_YEAR = build_topic_terms_table_figure(
    region_year_table,
    'Topic Terms Table for Year Created and Region',
)
scatter_fig_region_year.show()
LDA_PCA_TABLE_REGION_YEAR.show()
LDA_PCA_FIG_REGION_YEAR = scatter_fig_region_year

REGION_TOPIC_TABLE = TOPIC_PCA_TABLES['region_compressed'][0][['topic_id', 'PC0', 'PC1', 'mean_doc_weight', 'region_compressed', 'top_terms']].sort_values('mean_doc_weight', ascending=False).head(20).copy()
REGION_TOPIC_TABLE = REGION_TOPIC_TABLE.rename(columns={'region_compressed': 'region'})

LDA_PCA_FIGS['region_compressed'].write_html(LDA_PCA_REGION_HTML, include_plotlyjs='cdn')
LDA_PCA_TABLE_FIGS['region_compressed'].write_html(LDA_PCA_REGION_TABLE_HTML, include_plotlyjs='cdn')
REGION_TOPIC_TABLE.to_csv(LDA_PCA_REGION_TABLE_CSV, index=False)

print('Saved region graph:', LDA_PCA_REGION_HTML)
print('Saved region table html:', LDA_PCA_REGION_TABLE_HTML)
print('Saved region table csv:', LDA_PCA_REGION_TABLE_CSV)
REGION_TOPIC_TABLE


Saved region graph: lda_pca_region.html
Saved region table html: lda_pca_region_table.html
Saved region table csv: lda_pca_region_table.csv


,topic_id,PC0,PC1,mean_doc_weight,region,top_terms
2,T02,-0.373693,0.379515,0.079318,Eastern Europe,law laws courts matters jurisdiction decisions...
13,T13,0.390793,0.331904,0.073307,Oceania,office member election members term years person
15,T15,-0.384855,0.332219,0.071653,North Africa,rights people freedoms law citizens principles...
18,T18,-0.417108,0.268018,0.063536,South America,development education law resources services h...
4,T04,1.148742,0.246048,0.060593,Caribbean,office person functions accordance advice appo...
0,T00,0.136733,0.079994,0.057705,Oceania,days period law date time day force
6,T06,-0.231775,0.084045,0.055221,South America,duties functions law exercise members accordan...
10,T10,-0.137212,0.013762,0.054429,Western Europe,members majority vote votes election candidate...
17,T17,0.003836,0.045927,0.053186,Asia,government service power law authority powers ...
19,T19,-0.184107,-0.022468,0.051390,Sub-Saharan Africa,members number session sessions referendum rep...


## Final Project Summary

This section prints the main LDA details you need for the final project notebook.

In [7]:
TOP5_TOPICS = TOPICS.head(5).copy()
TOP5_TOPICS['top_5_words'] = TOP5_TOPICS['top_terms'].str.split().str[:5].str.join(', ')
TOP5_TOPICS['best_guess_label'] = ''

LDA_PROJECT_INFO = pd.Series({
    'topic_table': LDA_TOPICS_PARQUET.name,
    'count_matrix_used_to_create_topics': LDA_DTM_PARQUET.name,
    'theta_table': LDA_THETA_PARQUET.name,
    'phi_table': LDA_PHI_PARQUET.name,
    'delimiter': 'parquet',
    'library_used_to_compute': 'scikit-learn',
    'filtering': 'Nouns only via POS regex ^NNS?$',
    'bag_level': ', '.join(BAG_COLS),
    'country_level_summary': 'mean of bag-level topic weights grouped by country_id' if BAG_COLS != ['country_id'] else 'not needed; model fit directly at constitution level',
    'number_of_components': N_TOPICS,
    'countvectorizer_max_df': MAX_DF,
    'countvectorizer_min_df': MIN_DF,
    'countvectorizer_max_features': MAX_FEATURES,
    'countvectorizer_stop_words': 'sklearn English stopwords + custom legal stop terms from build_vocab',
    'lda_random_state': 42,
})

print('LDA project info:')
display(LDA_PROJECT_INFO.to_frame('value'))

print('Top five topics by mean document weight:')
display(TOP5_TOPICS[['top_5_words', 'doc_weight_mean', 'dominant_doc_count', 'best_guess_label']])

LDA project info:


,value
topic_table,lda_topics.parquet
count_matrix_used_to_create_topics,lda_dtm.parquet
theta_table,lda_theta.parquet
phi_table,lda_phi.parquet
delimiter,parquet
library_used_to_compute,scikit-learn
filtering,Nouns only via POS regex ^NNS?$
bag_level,"country_id, article_n"
country_level_summary,mean of bag-level topic weights grouped by cou...
number_of_components,20


Top five topics by mean document weight:


,top_5_words,doc_weight_mean,dominant_doc_count,best_guess_label
topic_id,,,,
T02,"law, laws, courts, matters, jurisdiction",0.079318,48,
T13,"office, member, election, members, term",0.073307,16,
T15,"rights, people, freedoms, law, citizens",0.071653,29,
T18,"development, education, law, resources, services",0.063536,24,
T04,"office, person, functions, accordance, advice",0.060593,30,


## Save LDA Tables

In [8]:
def prepare_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    # Flatten index levels safely so parquet exports do not collide with term columns.
    data_df = df.copy()
    if isinstance(data_df.index, pd.MultiIndex):
        index_df = data_df.index.to_frame(index=False)
        raw_names = list(data_df.index.names)
    else:
        index_name = data_df.index.name if data_df.index.name is not None else 'index_0'
        index_df = pd.DataFrame({index_name: data_df.index.to_numpy()})
        raw_names = [data_df.index.name]
    safe_names = []
    used_names = set(data_df.columns)
    for i, name in enumerate(raw_names):
        base_name = name if name is not None else f'index_{i}'
        safe_name = base_name
        if safe_name in used_names:
            safe_name = f'bag_{base_name}'
        while safe_name in used_names or safe_name in safe_names:
            safe_name = f'bag_{safe_name}'
        safe_names.append(safe_name)
    index_df.columns = safe_names
    data_df = data_df.reset_index(drop=True)
    return pd.concat([index_df, data_df], axis=1)


prepare_for_parquet(DTM).to_parquet(LDA_DTM_PARQUET, index=False)
prepare_for_parquet(THETA).to_parquet(LDA_THETA_PARQUET, index=False)
PHI.to_parquet(LDA_PHI_PARQUET)
TOPICS.to_parquet(LDA_TOPICS_PARQUET)

print('Saved files:')
print('-', LDA_DTM_PARQUET)
print('-', LDA_THETA_PARQUET)
print('-', LDA_PHI_PARQUET)
print('-', LDA_TOPICS_PARQUET)

Saved files:
- lda_dtm.parquet
- lda_theta.parquet
- lda_phi.parquet
- lda_topics.parquet
